# Titanic 



### Przygotowanie zbioru danych


In [1]:
import arff
import numpy as np
import pandas as pd

with open('Zbiór danych Titanic.arff', 'r') as f:
    data = arff.load(f)

df = pd.DataFrame(
    data['data'],
    columns=[attr[0] for attr in data['attributes']]
)

wiek_dziesietny = pd.concat([df.loc[df['age'].notna() & (df['age'] % 1 != 0), 'age']], axis=1)
wiek_dziesietny.columns = ["Wiek"]

display(df.head(20))
display(df.describe())
display(wiek_dziesietny.head(10))

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1.0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0.0,0.0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1.0,1,"Allison, Master. Hudson Trevor",male,0.9167,1.0,2.0,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1.0,0,"Allison, Miss. Helen Loraine",female,2.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1.0,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1.0,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
5,1.0,1,"Anderson, Mr. Harry",male,48.0000,0.0,0.0,19952,26.5500,E12,S,3,NaN,"New York, NY"
6,1.0,1,"Andrews, Miss. Kornelia Theodosia",female,63.0000,1.0,0.0,13502,77.9583,D7,S,10,NaN,"Hudson, NY"
7,1.0,0,"Andrews, Mr. Thomas Jr",male,39.0000,0.0,0.0,112050,0.0000,A36,S,NaN,NaN,"Belfast, NI"
8,1.0,1,"Appleton, Mrs. Edward Dale (Charlotte Lamson)",female,53.0000,2.0,0.0,11769,51.4792,C101,S,D,NaN,"Bayside, Queens, NY"
9,1.0,0,"Artagaveytia, Mr. Ramon",male,71.0000,0.0,0.0,PC 17609,49.5042,NaN,C,NaN,22.0,"Montevideo, Uruguay"


,pclass,age,sibsp,parch,fare,body
count,1309.000000,1046.000000,1309.000000,1309.000000,1308.000000,121.000000
mean,2.294882,29.881135,0.498854,0.385027,33.295479,160.809917
std,0.837836,14.413500,1.041658,0.865560,51.758668,97.696922
min,1.000000,0.166700,0.000000,0.000000,0.000000,1.000000
25%,2.000000,21.000000,0.000000,0.000000,7.895800,72.000000
50%,3.000000,28.000000,0.000000,0.000000,14.454200,155.000000
75%,3.000000,39.000000,1.000000,0.000000,31.275000,256.000000
max,3.000000,80.000000,8.000000,9.000000,512.329200,328.000000


,Wiek
1,0.9167
173,32.5000
222,28.5000
224,45.5000
359,0.8333
427,0.6667
512,32.5000
516,36.5000
548,0.8333
568,18.5000


W zbiorze Titanic jest 14 cech. Obejmują one informacje o:
- pclass - klasie biletu pasażera (1/2/3 klasa)
- survived - czy przeżył (binarne 1/0, 1-przeżył, 0-nie przeżył)
- name - informacje o pasażerze w formacie (nazwisko rodzinne, Tytuł. imię)
- sex - płeć pasażera (female/male)
- age - wiek pasażera (jak można zobaczyć poniżej pierwszej tabeli przypadki gdzie wiek nie jest liczbą całkowitą. Jeden gdzie wiek jest mniejszy od 1 - prawdopodobnie dzieci, które nie ukończyły pierwszego roku życia oraz przypadki, gdzie wiek jest w formacie __.5 - prawdopodobnie są to wartości estymowane)
- sibsp - ilość rodzeństwa pasażera na statku 
- parch - liczba rodziców lub dzieci pasażera na statku
- ticket - numer biletu pasażera
- fare - cena biletu pasażera (z drugiej tabeli możemy zauważyć, że najtańszy bilet/y był za 0 dolarów - prawdopodobnie będą to osoby powiązane ze statkiem np. właściciel albo osoby specjalne, które dostały zaproszenie)
- cabin - numer kabiny pasażera
- embarked - miejsce, w którym pasażer wsiadł na statek
- boat - identyfikator łodzi, którą uratowano pasażera
- body - identyfikator zwłok przypisany pasażerowi, który nie przeżył
- home.dest - informacje o miejscu zamieszkania i miejscu docelowym pasażera

### Brakujące wartości

W pliku ARFF braki danych są oznaczane symbolem '?'. Podczas wczytywania danych za pomocą biblioteki *arff* zostały one automatycznie zinterpretowane jako NaN. W związku z tym nie ma konieczności modyfikowania danych przed użyciem funkcji sprawdzających braki w zbiorze danych.

In [2]:
print(f"Ilość '?' w zbiorze danych: {(df == "?").sum().sum()}")
brak_ilo = df.isnull().sum()
brak_pro = df.isnull().mean() * 100 
display(pd.concat(
    [brak_ilo.rename('Ilość brakujących danych'), 
     brak_pro.rename('Procent brakujących danych [%]')], axis=1))

Ilość '?' w zbiorze danych: 0


,Ilość brakujących danych,Procent brakujących danych [%]
pclass,0,0.000000
survived,0,0.000000
name,0,0.000000
sex,0,0.000000
age,263,20.091673
sibsp,0,0.000000
parch,0,0.000000
ticket,0,0.000000
fare,1,0.076394
cabin,1014,77.463713


In [3]:
dfnull = df.copy()

for col in brak_ilo[brak_ilo > 0].index:
    if(f'{col}Null' not in df.columns):
        df[f"{col}Null"] = np.where(df[col].isnull(), 1, 0)
        #Jedna linijka
        # display(df.groupby('survived')[col].apply(lambda x: x.isnull().mean()))

display(df)

df_map = pd.DataFrame()
for col in df.columns[len(dfnull.columns):]:
    df_map = pd.concat([df_map, df.groupby('survived')[col].mean() * 100], axis=1)
display(df_map)


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,...,boat,body,home.dest,ageNull,fareNull,cabinNull,embarkedNull,boatNull,bodyNull,home.destNull
0,1.0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0.0,0.0,24160,211.3375,B5,...,2,NaN,"St Louis, MO",0,0,0,0,0,1,0
1,1.0,1,"Allison, Master. Hudson Trevor",male,0.9167,1.0,2.0,113781,151.5500,C22 C26,...,11,NaN,"Montreal, PQ / Chesterville, ON",0,0,0,0,0,1,0
2,1.0,0,"Allison, Miss. Helen Loraine",female,2.0000,1.0,2.0,113781,151.5500,C22 C26,...,NaN,NaN,"Montreal, PQ / Chesterville, ON",0,0,0,0,1,1,0
3,1.0,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1.0,2.0,113781,151.5500,C22 C26,...,NaN,135.0,"Montreal, PQ / Chesterville, ON",0,0,0,0,1,0,0
4,1.0,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1.0,2.0,113781,151.5500,C22 C26,...,NaN,NaN,"Montreal, PQ / Chesterville, ON",0,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,3.0,0,"Zabour, Miss. Hileni",female,14.5000,1.0,0.0,2665,14.4542,NaN,...,NaN,328.0,NaN,0,0,1,0,1,0,1
1305,3.0,0,"Zabour, Miss. Thamine",female,NaN,1.0,0.0,2665,14.4542,NaN,...,NaN,NaN,NaN,1,0,1,0,1,1,1
1306,3.0,0,"Zakarian, Mr. Mapriededer",male,26.5000,0.0,0.0,2656,7.2250,NaN,...,NaN,304.0,NaN,0,0,1,0,1,0,1
1307,3.0,0,"Zakarian, Mr. Ortin",male,27.0000,0.0,0.0,2670,7.2250,NaN,...,NaN,NaN,NaN,0,0,1,0,1,1,1


,ageNull,fareNull,cabinNull,embarkedNull,boatNull,bodyNull,home.destNull
0,23.485785,0.123609,87.391842,0.0,98.887515,85.043263,50.803461
1,14.600000,0.000000,61.400000,0.4,4.600000,100.000000,30.600000


### Rodzaje brakujących danych

- ageNull - *MAR* - prawdopodobieństwo nieznania wieku, mogło wynikać z klasy biletu.
- fareNull - *MCAR* - bardzo mała wartość braku danych, raczej jest to pojedyncze przeoczenie.
- cabinNull - *MAR* - brak numeru kabiny jest skorelowany z klasą biletu, ceną biletu i przeżywalnością - osoby z wyższą klasą biletu oraz społeczną miały większe szanse na ratunek.
- embarkedNull - *MCAR* - bardzo mała wartość braku danych, są to pojedyncze przeoczenia.
- boatNull - *MNAR* - wartość w tej kolumnie zależy od tego czy osoba skorzystała z łodzi ratunkowej, która de-facto w dużym stopniu mówi o tym czy ta osoba przeżyła.
- bodyNull - *MNAR* - wartość w tej kolumnie zależy od tego czy osoba przeżyła.
- home.destNull - *MAR* - informacje o miejscu docelowym zbierano od osób wyższych klas i tych które przeżyły.

### Co zrobić z brakującymi danymi

- age - uzupełnienie związane z innymi cechami np. name - w tej cesze mamy informację o tytule osoby, więc możemy je uzupełnić np. medianą danego tytułu
- fare - jeden przypadek, gdzie nie ma tej informacji; można usunąć wiersz - nie utrudni to dalszej analizy 
- cabin - dużo braków; można dla wierszy gdzie mamy informacje o kabinie zostawić pierwszą literę dla łatwiejszej kategoryzacji, a w przypadku braku zastąpić to np. literą "U" (Unknown)
- embarked - dwa przypadki, gdzie nie ma tej informacji; można usunąć wiersze - nie utrudni to dalszej analizy 
- boat - usunięcie całej kolumny; informacja bezpośrednio związana z przeżyciem - nie nadaje się do modelowania, zaburzyłaby cel modelu
- body - usunięcie całej kolumny; informacja bezpośrednio związana z przeżyciem - nie nadaje się do modelowania, zaburzyłaby cel modelu
- home.dest - dużo braków; można dla wierszy gdzie mamy informacje o destynacji zostawić tylko państwo, a w przypadku braku zastąpić to np. literą "U" (Unknown)